In [ ]:
## Nikolay Vorontsov, 23.11.2024, Prompting LLMs with validation set, ask to find hallucinations and label them.
## Model used: gemini-1.5-flash
## required files: llabel_validation_set_with_llm_for_baseline.env
##                 mushroom.en-val.v2.unlabeled.jsonl

In [ ]:
#INSTALL DEPENDENCIES

!pip install openai

In [3]:
# IMPORT LIBRARIES

import configparser
import json
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')

import time
#from openai import OpenAI
import google.generativeai as genai

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# DEFINE VARIABLES
#client = OpenAI(api_key = userdata.get('GPT_MUSHROOM'))

model_used = "gemini-1.5-flash"

Your_API_Key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=Your_API_Key)

model = genai.GenerativeModel(model_name=f"{model_used}")

prompts = configparser.ConfigParser()

prompts.read('/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_GPT.env')
output_file=f"/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_{model_used}.jsonl"

set_to_label =  '/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/mushroom.en-val.v2.unlabeled.jsonl'

In [5]:
# "a" creates the file if it doesn't exist
with open(output_file, "a") as file:
    pass  # Do nothing, just ensure the file exists

print(f"{output_file} is created or already exists.")

/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gemini-1.5-flash.jsonl is created or already exists.


In [6]:
# Function to load the .jsonl file and parse its contents
def load_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        # Read each line and parse as a JSON object
        return [json.loads(line) for line in file]

# Load the data
data = load_jsonl(set_to_label)


In [ ]:
#FUNCTIONS

In [7]:
# COMPILE PROMPTS
def define_prompt(datapoint):

  #samples can be also imported from a jsonl file.
  Sample1 = prompts.get('SAMPLES', 'Sample1')

  prompt1 = (
      f"{prompts.get('PROMPTS', 'p0')}"
      f"{datapoint}"
      f"{prompts.get('PROMPTS', 'p1')}"
      f"{prompts.get('PROMPTS', 'p2')}"
      f"{prompts.get('PROMPTS', 'p3')}"
      f"{prompts.get('PROMPTS', 'p4')}"
      f"{prompts.get('PROMPTS', 'p5')}"
      )

  return prompt1

In [8]:
define_prompt(data[1])

'This json line contains output from an llm, question-answer pair in "model_input", "model_output_text".{\'id\': \'val-en-2_unlabeled\', \'lang\': \'EN\', \'model_input\': \'How many genera does the Erysiphales order contain?\', \'model_output_text\': \'The Elysiphale order contains 5 genera.\', \'model_id\': \'tiiuae/falcon-7b-instruct\', \'soft_labels\': [], \'hard_labels\': [], \'model_output_logits\': [-6.199614048, -13.7564926147, -14.0058326721, -17.6579284668, -12.7987499237, -9.0510673523, -7.8389821053, -11.1029033661, -8.5361289978, -11.487569809, -8.2251300812, -9.1043262482], \'model_output_tokens\': [\'The\', \'ĠE\', \'lys\', \'iph\', \'ale\', \'Ġorder\', \'Ġcontains\', \'Ġ\', \'5\', \'Ġgenera\', \'.\', \'<|endoftext|>\']}Find all possible hallucinations (words/parts of text) in the "model_output_text", based on the knowledge from the internet. Select minimal hallucinations for the text to be correct.List each hallucination on a separate line.Do not add any extra symbols, 

In [22]:
def process_data(datapointX):
    selected_prompt = define_prompt(datapointX)
    #
    #completion = client.chat.completions.create(model=f"{model_used}",messages=[{"role": "user", "content": selected_prompt}])
    completion = model.generate_content(selected_prompt)
    #print(completion.text)
    hallucinated_words = [list_element for list_element in completion.text.split("\n") if list_element] ## this ensure list_element is not empty
    return hallucinated_words

In [23]:
## TEST process_data(datapoint)
x = process_data(data[20])

In [24]:
print(x)

for i in x:
  print(i)

['The Moscow Kremlin is a historic fortified complex at the heart of Moscow, Russia, which has served as the main residence of the Russian rulers since the 14th century.', 'Detinets, on the other hand, was the original fortified core of the Moscow Kremlin.', 'It was built in the late 13th century and served as the residence of the Grand Prince of Moscow and his court.', "It is a smaller, more compact area within the larger Kremlin complex, which includes the Cathedral Square with its famous cathedrals such as St. Basil's Cathedral and the Archangel Michael Cathedral."]
The Moscow Kremlin is a historic fortified complex at the heart of Moscow, Russia, which has served as the main residence of the Russian rulers since the 14th century.
Detinets, on the other hand, was the original fortified core of the Moscow Kremlin.
It was built in the late 13th century and served as the residence of the Grand Prince of Moscow and his court.
It is a smaller, more compact area within the larger Kremlin 

In [25]:
def find_spans(datapointX, hallucinated_words):

  spans = []
  model_output_text = datapointX["model_output_text"]

  for word in hallucinated_words:
      # Initialize the starting index for each word search
      start_index = 0
      while True:
          start_index = model_output_text.find(word, start_index)
          if start_index == -1:
              break
          end_index = start_index + len(word)
          spans.append([start_index, end_index])
          # Move the starting index past the current word to avoid overlapping results
          start_index = end_index

  return spans


In [26]:
## Assessing the output file

def last_line_number(file_path):
  # Read the last line of the file
  last_line = None
  with open(file_path, "r") as file:
      for line in file:
          last_line = line.strip()  # Store the current line

  # Parse the JSON object from the last line
  if last_line:
      last_data = json.loads(last_line)
      #print("Last JSON object:", last_data)
      return last_data["number"]
  else:
      print("The file is empty")
      return None


In [27]:
##TESTING
last_line_number(output_file)

The file is empty


In [28]:
for key, value in data[0].items():
  print(f'"{key}":') #type(value))

"id":
"lang":
"model_input":
"model_output_text":
"model_id":
"soft_labels":
"hard_labels":
"model_output_logits":
"model_output_tokens":


In [32]:
def label_and_save_data(data):
    processed_count = 0  # Counter for newly processed entries

    for number, datapoint in enumerate(data, start=1):
        # Get the last processed ID from the output file
        last_processed_number = last_line_number(output_file) or 0
        #print("Last processed ID:", last_processed_number)

        # Skip already processed entries
        if number <= last_processed_number:
            continue

        # Define prompt and process data
        prompt = define_prompt(datapoint)
        hallucinated_words = process_data(datapoint)
        hard_labels = find_spans(datapoint, hallucinated_words)

        # Save the datapoint to the JSONL file
        with open(output_file, "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "number": number,
                "id": datapoint["id"],
                "lang": datapoint["lang"],
                "model_input": datapoint["model_input"],
                "model_output_text": datapoint["model_output_text"],
                "model_id": datapoint["model_id"],
                "hallucinated_words": hallucinated_words,
                "soft_labels": [],
                "hard_labels": hard_labels,
                "model_output_logits": datapoint["model_output_logits"],
                "model_output_tokens": datapoint["model_output_tokens"],
            }

            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")
        print(datapoint["model_output_text"])
        print(hallucinated_words)
        print(hard_labels)
        print("-------------------------------")

        # Increment the processed count
        processed_count += 1

        ## Stop processing after 5 new entries, wait for 10 seconds
        if processed_count >= 5:
            print(f"Processed {processed_count} entries. Waiting for 10 sec. at ID {number}.")
            time.sleep(10)

            processed_count = 0


In [35]:
label_and_save_data(data)

In [36]:
output_file

'/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gemini-1.5-flash.jsonl'

In [37]:
## Resave a copy with no extra keys.

with open(output_file, "r", encoding='utf-8') as jsonl_file:
    lines = jsonl_file.readlines()

    for line in lines:

        # Remove '_unlabeled' from the 'id' field
        data_to_resave = json.loads(line)

        data_to_resave['id'] = data_to_resave['id'].replace('_unlabeled', '')

        print(data_to_resave["hard_labels"])

        soft_labels = [{'start': label[0], 'prob': float(1), 'end': label[1]} for label in data_to_resave['hard_labels'] if label]
        print(soft_labels)

        # Save the datapoint to the JSONL file
        with open(f"mushroom.en-val.v2.unlabeled.labelled_with_{model_used}_no_extra_keys_soft_labels_prob1.jsonl", "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "id":data_to_resave["id"],
                "lang":data_to_resave["lang"],
                "model_input":data_to_resave["model_input"],
                "model_output_text":data_to_resave["model_output_text"],
                "model_id":data_to_resave["model_id"],
                "soft_labels":soft_labels, #instead of data_to_resave["soft_labels"], that is to output an empty list.
                "hard_labels":data_to_resave["hard_labels"],
                "model_output_logits":data_to_resave["model_output_logits"],
                "model_output_tokens":data_to_resave["model_output_tokens"],
            }
            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")



[[25, 37], [45, 83]]
[{'start': 25, 'prob': 1.0, 'end': 37}, {'start': 45, 'prob': 1.0, 'end': 83}]
[[4, 14], [30, 31]]
[{'start': 4, 'prob': 1.0, 'end': 14}, {'start': 30, 'prob': 1.0, 'end': 31}]
[[9, 18], [24, 32]]
[{'start': 9, 'prob': 1.0, 'end': 18}, {'start': 24, 'prob': 1.0, 'end': 32}]
[[29, 33]]
[{'start': 29, 'prob': 1.0, 'end': 33}]
[[0, 36], [68, 228], [229, 231]]
[{'start': 0, 'prob': 1.0, 'end': 36}, {'start': 68, 'prob': 1.0, 'end': 228}, {'start': 229, 'prob': 1.0, 'end': 231}]
[[302, 362], [175, 229], [238, 293], [101, 170]]
[{'start': 302, 'prob': 1.0, 'end': 362}, {'start': 175, 'prob': 1.0, 'end': 229}, {'start': 238, 'prob': 1.0, 'end': 293}, {'start': 101, 'prob': 1.0, 'end': 170}]
[[53, 65], [87, 90]]
[{'start': 53, 'prob': 1.0, 'end': 65}, {'start': 87, 'prob': 1.0, 'end': 90}]
[[4, 11], [50, 57]]
[{'start': 4, 'prob': 1.0, 'end': 11}, {'start': 50, 'prob': 1.0, 'end': 57}]
[[61, 79]]
[{'start': 61, 'prob': 1.0, 'end': 79}]
[[10, 19], [20, 24]]
[{'start': 10, '